# Project 1 ANSC 4040 Data Split

This is a notebook for Project 1 that will be used for making the test, train and validation data split. The plan outlined below organizes the split chronologically by date and by cow. With the use of the tree-based model I plan to use, it it vital that all test cows are prepresented in the training at least once in order to improve the accuract of the model. 

In [1]:
#import pandas as pd
import pandas as pd

In [2]:
# Add file path and load the dataset
path = 'clean_data.csv'
data = pd.read_csv(path)
data.head()

,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-8.839529e+18,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5.365332e+18,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,NaN,NaN,NaN,NaN,2020-08-24,3.220506,3.501733,7.547777,14.560315,270,1
3,7.750424e+18,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
4,NaN,NaN,NaN,NaN,2021-03-21,4.218409,5.098378,9.198853,13.970645,198,2


# Plan for the split

1. Exlude missing values into a new data set- "missing_data"
2. Sort by date and by cow
3. Split by time order, ensuring each cow present in the test set is represented in the training set
     Keep the most recent dates in the test set (10%)
     Use earlier dates for training (70%)
     Use a validation window between them (20%)

# Exlude missing values into a new data set - "missing_data"

In [3]:
# Remove rows with any missing values from the clean dataset
before_rows = len(data)

data = data.dropna().copy()

after_rows = len(data)

print(f"Rows before removing missing values: {before_rows}")
print(f"Rows after removing missing values: {after_rows}")
print(f"Rows removed: {before_rows - after_rows}")

# Keep rows with missing values in a separate dataset
missing_data = data[data.isna().any(axis=1)].copy()

# Keep only rows without missing values
clean_data = data.dropna().copy()

missing_data.to_csv("missing_data.csv", index=False)
clean_data.to_csv("clean_data_no_missing.csv", index=False)


Rows before removing missing values: 8285371
Rows after removing missing values: 6599539
Rows removed: 1685832


KeyboardInterrupt: 

# Sort Remaining Data by Date and by COW
Keep the most recent dates in the test set (10%)
Use earlier dates for training (70%)
Use a validation window between them (20%)

ENSURING EACH COW FROM TEST SPLIT IS REPRESENTED IN TRAINING

In [4]:
# Add file path and load the dataset without any mising values
path = 'clean_data_no_missing.csv'
data = pd.read_csv(path)
data.head()

,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-8.839529e+18,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5.365332e+18,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,7.750424e+18,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
3,5.803650e+18,1.0,394.0,Open,2019-12-14,3.492661,4.399846,8.173735,11.793402,198,1
4,-1.713215e+18,5.0,288.0,Pregnant,2020-05-11,3.311224,4.300056,7.466130,8.572896,153,2


In [8]:
# Sort each cow's records chronologically before splitting by row count.
data = data.sort_values([cow_col, date_col]).reset_index(drop=True)

row_in_cow = data.groupby(cow_col, sort=False).cumcount()
rows_per_cow = data.groupby(cow_col, sort=False)[cow_col].transform("size")

# Keep at least one row per cow in training; other cutoffs are approximate
# when a cow has very few records.
train_end = (rows_per_cow * 0.70).round().astype(int).clip(lower=1)
val_end = (rows_per_cow * 0.90).round().astype(int)
val_end = val_end.where(val_end >= train_end, train_end)

train = data[row_in_cow < train_end].copy()
val = data[(row_in_cow >= train_end) & (row_in_cow < val_end)].copy()
test = data[row_in_cow >= val_end].copy()

total_rows = len(data)
print(f"Total rows: {total_rows:,}")
print(f"Train rows: {len(train):,} ({len(train) / total_rows:.1%})")
print(f"Validation rows: {len(val):,} ({len(val) / total_rows:.1%})")
print(f"Test rows: {len(test):,} ({len(test) / total_rows:.1%})")
print(f"Rows in splits: {len(train) + len(val) + len(test):,}")
print(f"All cows represented in training: {data[cow_col].nunique() == train[cow_col].nunique()}")

train.head()

Total rows: 6,462,990
Train rows: 4,523,957 (70.0%)
Validation rows: 1,292,705 (20.0%)
Test rows: 646,328 (10.0%)
Rows in splits: 6,462,990
All cows represented in training: True


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-9.221543e+18,4.0,85.0,Bred,2019-06-15,4.082331,4.799007,9.298644,18.688006,273,1
1,-9.221543e+18,4.0,85.0,Bred,2019-06-15,4.082331,4.998588,9.448329,14.469597,207,3
2,-9.221543e+18,4.0,85.0,Bred,2019-06-15,4.399846,5.198169,9.847490,18.189054,243,2
3,-9.221543e+18,4.0,86.0,Bred,2019-06-16,4.309128,4.998588,9.620694,16.918995,233,3
4,-9.221543e+18,4.0,86.0,Bred,2019-06-16,4.309128,5.297959,10.047071,18.642646,258,2


In [9]:
# Save the train, validation, and test splits as separate CSV files
train.to_csv("train_data.csv", index=False)
val.to_csv("val_data.csv", index=False)
test.to_csv("test_data.csv", index=False)

print("Saved: train_data.csv, val_data.csv, test_data.csv")

Saved: train_data.csv, val_data.csv, test_data.csv
